# Model Training & Evaluation

## Objective

The goal of this notebook is to train, evaluate and compare multiple machine learing models for customer churn prediction, we will use preprocessing pipeline built in the previous notebook to ensure a consistent and reproducible workflow. Model performance will be evaluat using multiple classification metrics, corss-validation, and hyperparameter tuning to identify the best-performing model. 

In [1]:
import pandas as pd
import numpy  as np


import os
import joblib
from joblib import load


from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

preprocessor = load("../models/preprocessor.joblib")

## Baseline Model

### business Question

Can a simple and  interpretable machine learning model accurately identify customers who are likely to churn?

To answer this question, we begin with Logistic Regresstion, a widely used baseline model for binary classification. Its porformance will serve as a benchmark for evaluating more complix models later in the project. 

In [3]:
log_reg_pipeline = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('classifier',LogisticRegression(random_state=42))
    ]
)

### Why a Pipeline?

Using a machine learning pipeline ensure that all preprocessing steps are applied consistently to both the training and testing datasets. It also prevents data leakage, improves reproducibility, and simplifiy the workflow by combining preprocessing and model training into a single object.

### Train the Model

In [ ]:
log_reg_pipeline.fit(X_train,y_train)

### Predictions

In [5]:
y_pred = log_reg_pipeline.predict(X_test)

y_prob_lr = log_reg_pipeline.predict_proba(X_test)[:, 1]

### Evaluate the Model

In [6]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob_lr)


metrics = pd.DataFrame({
    'Metric':[
        'Accuracy',
        'Precision',
        'Recall',
        'F1-score',
        'ROC_AUC'
    ],
    'Score':[
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

metrics

,Metric,Score
0,Accuracy,0.800568
1,Precision,0.655518
2,Recall,0.524064
3,F1-score,0.582467
4,ROC_AUC,0.842342


### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)


plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn'],
)

plt.xlabel("Predicted")
plt.ylabel('Actual')
plt.title('Confusion Matrix - Logistic Regression')

plt.show()

In [8]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409



## Model Interpretation

The logistic Regression provides a strong baseline for customer churn prediction. Rather than relying on a single metric, we evaluat the model using multiple performence measures.

- **Accuracy** measures the overall proportion of correctly classified customers.
- **Precision** indicates how many customers predicted to churn actually churned.
- **Recall** measures the model's ability to identify customers who truly churned.
- **F1-score** balances precision and recall, making it particularly useful for imbalanced datasets. 
- **ROC-AUC** evaluate the model's ability to distinguish between customers who churn and those who remain, regardless of the chosen classification threshold.

These metrics provide a comprehensive understanding of the model's strengths and limitations and establish baseline for compration with more advance machine learning algorithms.


## Baseline Model Analysis

Logistic Regression achieved an overall accuracy of **80%**, providing a soild baseline for customer churn predition. The model achieved a **ROC-AUC score of 0.842**, indicating good distinguish customers who churn and those who remain.

However the **recall of the churn class was 52.4%**, meaning nearly half of the customers who actually churned were not indentified by the model. In the customer retention setting, these false negative represent missed opportunities to intervene before customers leave.

Overall, Logistic Regression serves as a strong and interpretable baseline model. More flexible algorithms, such as Desision Trees or Random Forests, may improve recall and better capture the complex relationships within the data.

## Summary 

IN this notebook, we trained our first machine learning model using Logistic Regression and evluate it performance using multiple classification metrics. This model serves as the baseline for the project and provides a benchmark against which more complex models will be compared.

In the next stage, we will train a Desision Tree classifier and evaluate whether a non-linear model can improve predictive performance.

## Decision Tree Classifier

### Business Question 

Can a non-liner model can improve customer churn prediction be capturing complex relationships between customer characteristics and churn behavior?

## Build the Desision Tree Pipeline

### Why?

A machine learning pipeline combine data preprocessing and model training into a single workflow. This ensure that the same preprocessing steps are consistently applied both during training and prediction, reducing the risk of data leakage and improving reproduciblity.

In this section, we use a **Decision Tree Classifier** as our first non-linear model to determine whether it can better capture complex customers behavior than Logistic Regression.

In [9]:
# Decision tree pipeline

decision_tree_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(
            random_state=42
        ))
    ]
)

## Train the Decision Tree Model

The Decision Tree model is trained using the training dataset. During the process, the pipeline first applies all preprocessing steps to the input features and then fits the Decision Tree Classifier to learn patterns associated with customer churn.

In [ ]:
decision_tree_pipeline.fit(X_train, y_train)

## Generate Predictions 

After training, the model is used to predicte customer churn for the test dataset. We generate both class predictions and class probabilities.

- **Class predictions** are used to evaluate metrics such as accuracy, precision, recall, and F1-score.
- **Class probabilities** are required to calculate the ROC-AUC score. 

In [11]:
y_pred_dt = decision_tree_pipeline.predict(X_test)
y_prob_dt = decision_tree_pipeline.predict_proba(X_test)[:, 1]

## Evaluate Model Performance

The Decision Tree model is evaluated using mulitple classification metrics. Since customer churn prediction is an inbalanced calssification problem, relying on a single metric may provide misleading assessment. Therefore, we evaluate the model using accuracy, precision, recall, F1-score, and ROC-AUC.

In [12]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

decision_tree_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy_dt,
        precision_dt,
        recall_dt,
        f1_dt,
        roc_auc_dt
    ]
})

decision_tree_metrics

,Metric,Score
0,Accuracy,0.735273
1,Precision,0.501326
2,Recall,0.505348
3,F1-score,0.503329
4,ROC-AUC,0.661375


In [ ]:
cm_dt = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_dt,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [14]:
print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1035
           1       0.50      0.51      0.50       374

    accuracy                           0.74      1409
   macro avg       0.66      0.66      0.66      1409
weighted avg       0.74      0.74      0.74      1409



## Model Interpretation

The default Decision Tree classifier achieved lower performance than Logistic Regression across all evaluation metrics. While Decision Trees are capable of capturing complex and non-linear relationships, the default model appears to have overfitted the training data, resulting in poor generalization to unseen customers.

In particular, the ROC-AUC score decreased substantially, indicating that the model was less effective at distinguishing between customers who churn and those who remain. Similarly, recall and precision were both lower than those of the baseline Logistic Regression model.

These results suggest that the default Decision Tree is not the most suitable model for this problem. However, Decision Trees often benefit significantly from hyperparameter tuning, which will be explored later in the project.



## Random Forest Classifier

### Business Question

Can an ensemble of Decision Trees improve customer churn prediction by reducing overfitting and providing better generalization than a single Decision Tree?

## Build the Random Forest Pipeline

### Why?

A Random Forest combines multiple Decision Trees to creat a more robust and accurate classifier. Each tree is trained on a random sample of training data and consider a random subset of features when making splits. This ensemble approach helps reduce overfitting and improves model's ability to generalize on unseen data.

In this section, we evaluate whether a Random Forest outperform both Logistic Regression and single Decision Tree for customer churn prediction.

In [15]:
random_forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

## Train the Random Forest Model

The Random Forest model is trained using the training dataset. During training, the preprocessing pipeline transforms the input features before fitting multiple Decision Trees. The final model combines the predictions of all trees through majority voting.

In [ ]:
random_forest_pipeline.fit(X_train, y_train)

## Generate Predictions

After training, the Random Forest model is used to predict customer churn on the test dataset. Both class predictions and predicted probabilities are generated for model evaluation.

In [17]:
y_pred_rf = random_forest_pipeline.predict(X_test)
y_prob_rf = random_forest_pipeline.predict_proba(X_test)[:, 1]

## Evaluate Model Performance

The Random Forest classifier is evaluated using multiple classification metrics to assess its predictive performance. Since customer churn prediction is an imbalanced classification problem, multiple evaluation metrics are considered to obtain a comprehensive assessment of the model.

In [18]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

random_forest_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy_rf,
        precision_rf,
        recall_rf,
        f1_rf,
        roc_auc_rf
    ]
})

random_forest_metrics

,Metric,Score
0,Accuracy,0.774308
1,Precision,0.595890
2,Recall,0.465241
3,F1-score,0.522523
4,ROC-AUC,0.821334


In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_rf,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [20]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.82      0.89      0.85      1035
           1       0.60      0.47      0.52       374

    accuracy                           0.77      1409
   macro avg       0.71      0.68      0.69      1409
weighted avg       0.76      0.77      0.76      1409



## Model Comparison

Now that we have trained and evaluated all three models individually, we bring their results together into a single comparison table. This makes it easier to judge each model's trade-offs across all metrics at once, rather than scrolling between separate outputs.


In [21]:
# Combine metrics from all three models into a single comparison table

comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Logistic Regression": [accuracy, precision, recall, f1, roc_auc],
    "Decision Tree": [accuracy_dt, precision_dt, recall_dt, f1_dt, roc_auc_dt],
    "Random Forest": [accuracy_rf, precision_rf, recall_rf, f1_rf, roc_auc_rf],
})

comparison_df.set_index("Metric").round(3)


,Logistic Regression,Decision Tree,Random Forest
Metric,,,
Accuracy,0.801,0.735,0.774
Precision,0.656,0.501,0.596
Recall,0.524,0.505,0.465
F1-score,0.582,0.503,0.523
ROC-AUC,0.842,0.661,0.821


In [ ]:
# Visualize the comparison as a grouped bar chart

comparison_melted = comparison_df.melt(id_vars="Metric", var_name="Model", value_name="Score")

plt.figure(figsize=(9, 5))
sns.barplot(data=comparison_melted, x="Metric", y="Score", hue="Model")

plt.title("Model Comparison Across Evaluation Metrics")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.legend(loc="lower right")
plt.show()

Although accuracy provides a useful overall measure of model performance, it is not sufficient for customer churn prediction because the dataset is imbalanced. Therefore, Recall and ROC-AUC are given greater importance when comparing models.

## Interpretation

**Logistic Regression remains the strongest model overall**, achieving the highest score on every metric. This is a notable result: although both the Decision Tree and Random Forest are more flexible, non-linear models capable of capturing feature interactions, neither outperformed the simpler linear baseline on this dataset.

A few factors likely explain this outcome:

- **Dataset size.** With only ~5,600 training rows, tree-based models — especially a fully-grown Decision Tree — have limited data to learn stable, generalizable splits. This leaves them prone to overfitting the training set rather than capturing the true underlying pattern.
- **Default hyperparameters.** Both tree-based models were trained with default settings (e.g., no `max_depth` constraint, default `n_estimators` for the forest). Without tuning, they are not yet operating at their best possible performance.
- **Random Forest still clearly improves on the Decision Tree.** Averaging across many trees substantially reduced overfitting: ROC-AUC rose from 0.661 to 0.821, and every other metric improved as well. This matches the expected effect of variance reduction through ensembling.
- **Random Forest did not surpass Logistic Regression.** Despite the improvement over the single tree, Random Forest still fell short of Logistic Regression on every metric. This suggests that, at least in its default form, the added model complexity is not (yet) translating into better generalization than the simpler linear model on this dataset.

**Conclusion:** the default Random Forest confirms the hypothesis for the Decision Tree comparison (ensembling clearly reduces variance and improves on a single tree) but does not confirm it for the Logistic Regression comparison. This isn't a final verdict — hyperparameter tuning (`GridSearchCV`) may close or reverse this gap, and that will be explored in the next stage of the project.


## Cross-Validation

A single train-test split provides a useful estimate of model performance, but the result can depend on how the data is divided. To obtain a more reliable estimate of model performance, we use stratified k-fold cross-validation.

Stratified cross-validation preserves the proportion of churn and non-churn observations within each fold, which is particularly important because the target variable is imbalanced.

In this project, we use 5-fold cross-validation to evaluate whether the baseline models perform consistently across different subsets of the training data.

In [23]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Cross-Validation Metrics

Because customer churn is an imbalanced classification problem, accuracy alone is not sufficient for evaluating model performance.

Recall is particularly important because failing to identify a customer who is likely to churn represents a missed opportunity for customer retention. However, precision, F1-score, and ROC-AUC are also considered to understand the model from different perspectives.

Therefore, we evaluate all four metrics during cross-validation rather than relying on a single score.

In [24]:
scoring = {
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [25]:
# Logistic Regression cross-validation

log_reg_cv = cross_validate(
    log_reg_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [26]:
# Look at the results
log_reg_cv_results = pd.DataFrame({
    "Metric": [
        "Recall",
        "Precision",
        "F1-score",
        "ROC-AUC"
    ],
    "Mean": [
        log_reg_cv["test_recall"].mean(),
        log_reg_cv["test_precision"].mean(),
        log_reg_cv["test_f1"].mean(),
        log_reg_cv["test_roc_auc"].mean()
    ],
    "Std": [
        log_reg_cv["test_recall"].std(),
        log_reg_cv["test_precision"].std(),
        log_reg_cv["test_f1"].std(),
        log_reg_cv["test_roc_auc"].std()
    ]
})

log_reg_cv_results.round(3)

,Metric,Mean,Std
0,Recall,0.532,0.036
1,Precision,0.675,0.022
2,F1-score,0.595,0.025
3,ROC-AUC,0.846,0.012


## Logistic Regression Cross-Validation Results

The Logistic Regression model achieved a mean Recall of 0.532, Precision of 0.675, F1-score of 0.595, and ROC-AUC of 0.846 across five stratified folds.

The ROC-AUC result is particularly consistent with the previous test-set evaluation, where the model achieved a ROC-AUC of 0.842. The small difference between the cross-validation and test-set results suggests that the model's ability to distinguish churners from non-churners is reasonably stable across different subsets of the data.

The standard deviations also provide insight into the model's consistency across folds. In particular, the ROC-AUC standard deviation of 0.012 indicates relatively low variation in ranking performance, while Recall shows somewhat greater variation with a standard deviation of 0.036.

Overall, the cross-validation results support Logistic Regression as a strong and relatively stable baseline model for this churn prediction task.


## Decision Tree Cross-Validation

The Decision Tree model is evaluated using the same 5-fold stratified cross-validation strategy and evaluation metrics used for Logistic Regression.

Using the same folds and metrics allows us to make a fair comparison between the models and determine whether the poor performance observed on the initial test split is consistent across different subsets of the training data.

In [27]:
decision_tree_cv = cross_validate(
    decision_tree_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [28]:
decision_tree_cv_results = pd.DataFrame({
    'Metric':[
        'Recall',
        'Precision',
        'F1-score',
        'ROC-AUC'
    ],
    'Mean':[
        decision_tree_cv['test_recall'].mean(),
        decision_tree_cv['test_precision'].mean(),
        decision_tree_cv['test_f1'].mean(),
        decision_tree_cv['test_roc_auc'].mean()
    ],
    'Std':[
        decision_tree_cv['test_recall'].std(),
        decision_tree_cv['test_precision'].std(),
        decision_tree_cv['test_f1'].std(),
        decision_tree_cv['test_roc_auc'].std()
    ]
})

decision_tree_cv_results.round(3)

,Metric,Mean,Std
0,Recall,0.496,0.028
1,Precision,0.489,0.025
2,F1-score,0.493,0.025
3,ROC-AUC,0.655,0.018


## Decision Tree Cross-Validation Results

The Decision Tree achieved a mean Recall of 0.496, Precision of 0.489, F1-score of 0.4493, and ROC-AUC of 0.655 across five stratified folds.

The cross-validation results are highly consistent with the initial test-set evaluation, where the model achieved a ROC-AUC of 0.661. The difference of only 0.006 suggests that the poor performance observed on the test set was not primarily caused by an unfavorable train-test split.

The relatively low ROC-AUC across all folds indicates that the default Decision Tree has limited ability to distinguish between churners and non-churners on this dataset. Its low performance also suggests that the default tree configuration may not be capturing the underlying patterns effectively.

However, this does not necessarily mean that Decision Trees are unsuitable for the problem. Hyperparameter tuning may improve the model by controlling its complexity and finding a more appropriate tree structure.

## Random Forest Cross-Validation

The Random Forest is evaluated using the same 5-fold stratified cross-validation strategy and evaluation metrics.

This allows us to determine whether the substantial improvement over the single Decision Tree observed on the initial test set is consistent across different subsets of the training data.

In [29]:
random_forest_cv = cross_validate(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

In [30]:
random_forest_cv_results = pd.DataFrame({
    'Metric':[
        'Recall',
        'Precision',
        'F1-score',
        'ROC-AUC'
    ],
    'Mean':[
        random_forest_cv['test_recall'].mean(),
        random_forest_cv['test_precision'].mean(),
        random_forest_cv['test_f1'].mean(),
        random_forest_cv['test_roc_auc'].mean(),
    ],
    'Std':[
        random_forest_cv['test_recall'].std(),
        random_forest_cv['test_precision'].std(),
        random_forest_cv['test_f1'].std(),
        random_forest_cv['test_roc_auc'].std(),
    ]
})

random_forest_cv_results.round(3)

,Metric,Mean,Std
0,Recall,0.485,0.013
1,Precision,0.638,0.037
2,F1-score,0.551,0.020
3,ROC-AUC,0.822,0.011


## Random Forest Cross-Validation Results

The Random Forest achieved a mean Recall of 0.485, Precision of 0.638, F1-score of 0.551, and ROC-AUC of 0.822 across five stratified folds.

These results are consistent with the initial test-set evaluation, where the model achieved a ROC-AUC of 0.821. The small difference between the cross-validation and test-set performance suggests that the model's performance is reasonably stable across different subsets of the training data.

Random Forest substantially outperformed the single Decision Tree, particularly in ROC-AUC, increasing from 0.655 to 0.822 in cross-validation. This supports the idea that combining multiple trees can reduce variance and improve generalization.

However, Random Forest still performed below Logistic Regression, which achieved a mean ROC-AUC of 0.846 and Recall of 0.532. Therefore, Logistic Regression remains the strongest baseline model at this stage.

In [31]:
# Making table for comparision all corss-validations
cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Recall": [
        f"{log_reg_cv['test_recall'].mean():.3f} ± {log_reg_cv['test_recall'].std():.3f}",
        f"{decision_tree_cv['test_recall'].mean():.3f} ± {decision_tree_cv['test_recall'].std():.3f}",
        f"{random_forest_cv['test_recall'].mean():.3f} ± {random_forest_cv['test_recall'].std():.3f}"
    ],
    "Precision": [
        f"{log_reg_cv['test_precision'].mean():.3f} ± {log_reg_cv['test_precision'].std():.3f}",
        f"{decision_tree_cv['test_precision'].mean():.3f} ± {decision_tree_cv['test_precision'].std():.3f}",
        f"{random_forest_cv['test_precision'].mean():.3f} ± {random_forest_cv['test_precision'].std():.3f}"
    ],
    "F1-score": [
        f"{log_reg_cv['test_f1'].mean():.3f} ± {log_reg_cv['test_f1'].std():.3f}",
        f"{decision_tree_cv['test_f1'].mean():.3f} ± {decision_tree_cv['test_f1'].std():.3f}",
        f"{random_forest_cv['test_f1'].mean():.3f} ± {random_forest_cv['test_f1'].std():.3f}"
    ],
    "ROC-AUC": [
        f"{log_reg_cv['test_roc_auc'].mean():.3f} ± {log_reg_cv['test_roc_auc'].std():.3f}",
        f"{decision_tree_cv['test_roc_auc'].mean():.3f} ± {decision_tree_cv['test_roc_auc'].std():.3f}",
        f"{random_forest_cv['test_roc_auc'].mean():.3f} ± {random_forest_cv['test_roc_auc'].std():.3f}"
    ]
})

cv_comparison

,Model,Recall,Precision,F1-score,ROC-AUC
0,Logistic Regression,0.532 ± 0.036,0.675 ± 0.022,0.595 ± 0.025,0.846 ± 0.012
1,Decision Tree,0.496 ± 0.028,0.489 ± 0.025,0.493 ± 0.025,0.655 ± 0.018
2,Random Forest,0.485 ± 0.013,0.638 ± 0.037,0.551 ± 0.020,0.822 ± 0.011


## Cross-Validation Model Comparison

The three baseline models were evaluated using the same 5-fold stratified cross-validation strategy. Mean performance is reported together with the standard deviation across folds to assess both model performance and consistency.

Because customer churn is an imbalanced classification problem and the business objective emphasizes identifying potential churners, Recall is given particular importance. ROC-AUC is also considered because it measures the model's ability to distinguish between churners and non-churners across different classification thresholds.

### Cross_Validation Interpretation

Logistic Regression achieved the strongest cross-validation performance across the evaluation metrics, with a mean recall of 0.532 and ROC-AUC OF 0.846. It also achieved the highest Precision and F1-score among the three baseline models.

The Decision Tree produce the weakest results, with a mean ROC-AUC of 0.655. It performance was also relatively consistent across folds, suggesting that its poor performace was not primarily caused by a particular train-validation split.

Random Forest substantially improved upon the single Decision Tree, this demonetrates the benefit of combining multiple decision trees to reduce variance and improve generalization. However, its recall of 0.485 remained below both Logistic Regression and the Decision Tree.

Overall, Logistic Regression is still the best baseline, both in terms of raw performance and how well it fits the actual business goal of catching customers who are likely to churn. That said, the tree-based models haven't been tuned yet, so I'll run some hyperparameter tuning to see if they can close the gap.

## Hyperparameter Tuning

The baseline models have now been evaluated using cross-validation. Logistic Regression performed best overall, while Random Forest was the strongest tree-based model.

In the next step, we will tune the Random Forest hyperparameters to see whether we can improve its performance, particularly its Recall. Since our goal is to identify as many customers likely to churn as possible, Recall will be used as the main metric for selecting the best model configuration.

In [ ]:
# Random Forset Grid Search parameters

rf_param_grid = {
    'classifier__n_estimators': [200, 400],
    'classifier__max_depth': [5, 10, 15, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt','log2']
}

In [ ]:
rf_grid_search = GridSearchCV(
    estimator= random_forest_pipeline,
    param_grid=rf_param_grid,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [ ]:
rf_grid_search.fit(X_train,y_train)

In [ ]:
tuned_rf = rf_grid_search.best_estimator_

joblib.dump(rf_grid_search, "rf_grid_search_result.pkl")
joblib.dump(tuned_rf, "best_rf_model.pkl")

In [33]:
best_rf_model = joblib.load("../models/best_rf_model.pkl")

In [35]:
rf_grid_search = joblib.load("../models/rf_grid_search_result.pkl")


print("Best Recall:", rf_grid_search.best_score_)
print("Best Parameters:")
print(rf_grid_search.best_params_)

Best Recall: 0.4996655518394649
Best Parameters:
{'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}


## Random Forest Tuning Results

The hyperparameter search improved the Random Forest's mean cross-validation Recall from 0.485 to 0.499.

The best configuration used a maximum tree depth of 10, 200 trees, and required at least 10 samples to split an internal node. These parameters were selected based on the highest Recall achieved during 5-fold cross-validation.

Although tuning improved the Random Forest, its Recall is still slightly below the Logistic Regression baseline, which achieved a cross-validation Recall of 0.532.

## Tuned Random Forest — Test Set Evaluation

We tuned the Random Forest using 5-fold cross-validation, focusing on Recall as the main metric. After tuning, the best model reached a mean CV Recall of 0.499, which is a noticeable improvement over the baseline score of 0.485.

Now the important question is whether this improvement actually holds up on data the model has never seen before. So in this step, we’ll evaluate the tuned model on the test set and check how it performs in practice.

Along with Recall, we’ll also look at Precision, F1-score, ROC-AUC, and Accuracy to get a full picture of how the tuning affected the model overall.


In [36]:
# get the best model selected by GridSearchCV
best_rf_model

# generate prediction on the test set
y_pred_tuned_rf = best_rf_model.predict(X_test)

# predict probabilities for the ROC-AUC
y_prob_tuned_rf = best_rf_model.predict_proba(X_test)[:, 1]

In [37]:
accuracy_tuned_rf = accuracy_score(y_test, y_pred_tuned_rf)
precision_tuned_rf = precision_score(y_test, y_pred_tuned_rf)
recall_tuned_rf = recall_score(y_test, y_pred_tuned_rf)
f1_tuned_rf = f1_score(y_test, y_pred_tuned_rf)
roc_auc_tuned_rf = roc_auc_score(y_test, y_prob_tuned_rf)

tuned_rf_metrics = pd.DataFrame({
    'Metric':[
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'ROC-AUC'
    ],
    'Score': [
        accuracy_tuned_rf,
        precision_tuned_rf,
        recall_tuned_rf,
        f1_tuned_rf,
        roc_auc_tuned_rf
    ]
})

tuned_rf_metrics

,Metric,Score
0,Accuracy,0.796309
1,Precision,0.653710
2,Recall,0.494652
3,F1,0.563166
4,ROC-AUC,0.841228


## Tuned Random Forest — Results

The tuned Random Forest improved noticeably over the baseline model. Test-set Recall increased from 0.465 to 0.494, which is consistent with the improvement observed during cross-validation (0.485 to 0.499). This suggests that the tuning improvement generalizes reasonably well to unseen data.

The other metrics also improved, with Precision increasing from 0.596 to 0.653 and ROC-AUC from 0.821 to 0.841. Overall, hyperparameter tuning produced a stronger Random Forest without sacrificing its other evaluation metrics.

However, Logistic Regression still has a higher cross-validation Recall (0.532), so we cannot consider the tuned Random Forest the final model yet. We will compare the models more carefully before making the final selection.

In [ ]:
cm_tuned_rf = confusion_matrix(y_test, y_pred_tuned_rf)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_tuned_rf,
    annot=True,
    fmt='d',
    xticklabels=['No Churn','Churn'],
    yticklabels=['No Churn','Churn']
)

plt.title('Tuned Random Forest Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True label')
plt.show()

The tuned Random Forest correctly identified 185 of the 374 customers who actually churned, resulting in a Recall of 49.5%. This means the model detected about half of the customers who eventually churned, while 189 churners were missed.

From a business perspective, these 189 false negatives represent potential missed retention opportunities. Since the main objective is to identify customers who are likely to churn so that the company can intervene, reducing false negatives is particularly important.

At the same time, the model produced 98 false positives, meaning some customers were incorrectly identified as potential churners. These customers could receive unnecessary retention offers or interventions. Therefore, the model involves a trade-off between identifying more churners and avoiding unnecessary interventions.

In [39]:
print(classification_report(y_test, y_pred_tuned_rf))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1035
           1       0.65      0.49      0.56       374

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.80      0.79      1409



## Classification Report

The classification report shows that the tuned Random Forest performs substantially better on the No Churn class than on the Churn class. It achieves a Recall of 0.91 for customers who remain, compared with 0.49 for customers who churn.

For the Churn class, the model has a Precision of 0.65, meaning that about two-thirds of customers predicted to churn actually churned. However, the Recall of 0.49 shows that the model still misses a significant proportion of actual churners.

The difference between the macro and weighted averages also highlights the effect of class imbalance. The weighted metrics are influenced more by the larger No Churn class, so they provide a more favorable overall picture than the performance on the Churn class alone.

Since the main business objective is to identify customers who are likely to churn, the Churn class metrics—particularly Recall—remain more important for evaluating this model.


## Model Comparison After Random Forest Tuning

After tuning the Random Forest, we can now compare its performance with the baseline Logistic Regression, Decision Tree, and Random Forest models.

Since the main objective is to identify customers who are likely to churn, Recall is particularly important. However, Precision, F1-score, and ROC-AUC are also considered to understand the trade-offs between identifying churners and generating incorrect predictions.


In [40]:
comparison_after_tuning = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        accuracy,
        accuracy_dt,
        accuracy_rf,
        accuracy_tuned_rf
    ],
    "Precision": [
        precision,
        precision_dt,
        precision_rf,
        precision_tuned_rf
    ],
    "Recall": [
        recall,
        recall_dt,
        recall_rf,
        recall_tuned_rf
    ],
    "F1-score": [
        f1,
        f1_dt,
        f1_rf,
        f1_tuned_rf
    ],
    "ROC-AUC": [
        roc_auc,
        roc_auc_dt,
        roc_auc_rf,
        roc_auc_tuned_rf
    ]
})

comparison_after_tuning.round(3)

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.801,0.656,0.524,0.582,0.842
1,Decision Tree,0.735,0.501,0.505,0.503,0.661
2,Random Forest,0.774,0.596,0.465,0.523,0.821
3,Tuned Random Forest,0.796,0.654,0.495,0.563,0.841


## Model Selection After Random Forest Tuning

After tuning the Random Forest, Logistic Regression remains the strongest candidate based on the current test-set results. It achieves a Recall of 0.524 compared with 0.495 for the tuned Random Forest, meaning it identifies slightly more of the customers who actually churn.

Logistic Regression also achieves a slightly higher F1-score and ROC-AUC, while the tuned Random Forest provides slightly higher Precision. Since the primary business objective is to identify as many potential churners as possible, the higher Recall makes Logistic Regression the preferred candidate at this stage.

However, this is not yet the final model selection. The current results use the default classification threshold, so we will next investigate whether adjusting the decision threshold can improve the Recall–Precision trade-off and produce a model that better matches the business objective.


## Exploratory Threshold Analysis

Before selecting a final classification threshold, we first examine how different thresholds affect the Precision–Recall trade-off.

This analysis is exploratory and uses the test set only to understand the behavior of the model. The results will not be used to make the final threshold-selection decision.


In [41]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob_lr >= threshold).astype(int)


    threshold_results.append({
        'Threshold': threshold,
        'Precision': precision_score(y_test, y_pred_threshold),
        'Recall': recall_score(y_test, y_pred_threshold),
        'F1-score': f1_score(y_test, y_pred_threshold)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(3)

,Threshold,Precision,Recall,F1-score
0,0.50,0.656,0.524,0.582
1,0.45,0.608,0.586,0.597
2,0.40,0.580,0.647,0.612
3,0.35,0.559,0.709,0.625
4,0.30,0.525,0.751,0.618


The results show a clear trade-off between Recall and Precision. As the threshold decreases from 0.50 to 0.30, Recall increases from 52.4% to 75.1%, while Precision decreases from 65.6% to 52.5%.

This suggests that lower thresholds may be more suitable for our business objective because they allow the model to identify more potential churners. However, the increase in Recall comes with more false positives. The final threshold will therefore be selected using cross-validation on the training data.


## Threshold Selection with Cross-Validation

The exploratory analysis showed that the classification threshold has a strong effect on Recall and Precision. To select the final threshold without using the test set, we will now evaluate different thresholds using 5-fold cross-validation on the training data.


In [42]:
y_prob_oof = cross_val_predict(
    log_reg_pipeline,
    X_train,
    y_train,
    cv= cv,
    method='predict_proba'
)[:, 1]

In [43]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

cv_threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob_oof >= threshold).astype(int)

    cv_threshold_results.append({
        'Threshold':threshold,
        'precision':precision_score(y_train, y_pred_threshold),
        'Recall':recall_score(y_train, y_pred_threshold),
        'F1-score':f1_score(y_train, y_pred_threshold)
    })

cv_threshold_df = pd.DataFrame(cv_threshold_results)

cv_threshold_df.round(3)

,Threshold,precision,Recall,F1-score
0,0.50,0.675,0.532,0.595
1,0.45,0.632,0.591,0.611
2,0.40,0.602,0.652,0.626
3,0.35,0.572,0.711,0.634
4,0.30,0.537,0.753,0.627


## Final Threshold Selection

The cross-validated threshold analysis on the training data confirms the pattern seen in the exploratory test-set analysis: lowering the threshold increases Recall at the cost of Precision. While 0.35 gives the best F1-score balance (0.634), we select **0.30** as the final threshold. Under our business assumption that missing a churner is more costly than an unnecessary retention contact, the Recall gain from 0.35 to 0.30 (+4.2 points) is judged to outweigh the modest Precision loss (-3.5 points), even though it comes with a small F1 trade-off (0.634 → 0.627).

## Final Model — Test Set Evaluation

We now fit the final Logistic Regression on the full training data and evaluate it once on the untouched test set, applying our selected threshold of 0.30. This is the definitive evaluation of the model as it would be used in production.

In [44]:
final_threshold = 0.30
y_pred_final = (y_prob_lr >= final_threshold).astype(int)

In [45]:
final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1_score = f1_score(y_test, y_pred_final)
final_roc_auc = roc_auc_score(y_test, y_prob_lr)


final_metrics = pd.DataFrame({
    'Metric':[
        'Accuracy',
        'Precision',
        'Recall',
        'F1-score',
        'ROC-AUC'
    ],

    'Score':[
        final_accuracy,
        final_precision,
        final_recall,
        final_f1_score,
        final_roc_auc
    ]
})

final_metrics.round(3)

,Metric,Score
0,Accuracy,0.754
1,Precision,0.525
2,Recall,0.751
3,F1-score,0.618
4,ROC-AUC,0.842


## Final Model — Confusion Matrix and Classification Report

Before interpreting the model, we look at the confusion matrix and classification report at threshold=0.30 to see exactly how predictions break down: how many churners we correctly catch, how many we miss, and how many false alarms we generate.

In [ ]:
cm_final = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_final,
    annot=True,
    fmt='d',
    xticklabels=['No Chrun', 'Churn'],
    yticklabels=['No Chrun', 'Churn']
)

plt.title('Final Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True Label')
plt.show()

In [47]:
print(cm_final)

[[781 254]
 [ 93 281]]


In [48]:
print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.89      0.75      0.82      1035
           1       0.53      0.75      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.75      0.77      1409



## Feature Importance and Interpretation

We now examine the Logistic Regression coefficients to understand which features most influence churn predictions.The coefficient sign indicates whether a feature is associated with higher or lower churn probability, while the coefficient magnitude provides a relative indication of its contribution within the model.

In [49]:
feature_names = log_reg_pipeline.named_steps['preprocessor'].get_feature_names_out()
coefficients = log_reg_pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", ascending=False)

coef_df

,Feature,Coefficient
20,cat__internet_service_Fiber optic,0.948546
34,cat__contract_Month-to-month,0.671178
45,cat__tenure_group_Loyal,0.392518
4,num__total_services,0.264307
41,cat__payment_method_Electronic check,0.208508
3,num__total_charges,0.178371
6,num__streaming_services,0.092514
22,cat__online_security_No,0.063022
38,cat__paperless_billing_Yes,0.061688
0,num__senior_citizen,0.056051


## Interpreting the Coefficients

After fixing the multicollinearity issue, the coefficients are now individually meaningful. The strongest churn-reducing signals are `tenure` and having no internet service at all, while the strongest churn-driving signals are Fiber optic internet and month-to-month contracts. This matches the earlier hypothesis about contract type and internet service, and also surfaces tenure as the dominant factor overall.

Notably, several coefficients changed after removing the redundant "No internet service" categories — most clearly `internet_service_No`, whose effect had previously been split across seven duplicate columns (each showing the same diluted -0.258) and is now visible as a single strong effect (-1.076). This confirms the multicollinearity fix meaningfully improved interpretability without changing predictive performance.

## Business Insights and Recommendations

Based on the model's coefficients and evaluation results, we translate the key churn drivers and churn reducers into actionable recommendations for the retention team.

**Churn drivers**

1. **Investigate Fiber Optic service quality and pricing.** Fiber optic internet is the strongest churn-associated feature. This does not tell us *why* customers leave, only that they do at a higher rate — the retention team should investigate whether the issue is service reliability, support quality, or price relative to competitors.

2. **Investigate the drivers behind month-to-month churn.** Month-to-month contracts are the second-strongest churn-associated feature. The team should determine whether this reflects price sensitivity, dissatisfaction, or simply the lower switching cost inherent to short-term contracts.

**Churn reducers**

3. **Incentivize longer-term contracts.** Tenure and two-year contracts are the strongest churn-reducing factors in the model. The retention team should target month-to-month customers — especially newer, high-risk ones — with incentives (e.g. a discount) to switch to a one- or two-year contract.

4. **Promote tech support and security add-ons at key moments.** Customers without tech support or online security show a modestly higher churn tendency. Rather than a blanket push, the team should bundle a trial of these services specifically for customers who already show other risk factors (e.g. Fiber optic + month-to-month), where the combined risk is highest.

**Operational use of the model**

5. **Score the customer base monthly and prioritize outreach by risk.** Beyond segment-level patterns, the model can output a churn probability for each individual customer. A customer with several "risky" traits may still be low-risk overall if offsetting factors (e.g. long tenure) are strong enough — something segment-level insights alone can't capture. The retention team should run the model monthly, rank customers by predicted churn probability, and focus outreach on the highest-risk segment rather than broad customer categories.

## Saving the Final Model

We save the trained Logistic Regression pipeline to disk using `joblib`, so it can be reloaded later for predictions without retraining. We also record the selected classification threshold (0.30), since it is not part of the saved pipeline object itself and must be applied manually at prediction time.

In [50]:
os.makedirs("../models", exist_ok=True)

joblib.dump(log_reg_pipeline, "../models/churn_model_logreg.pkl")

with open("../models/model_metadata.txt", "w") as f:
    f.write("Model: Logistic Regression\n")
    f.write("Selected threshold: 0.30\n")
    f.write(f"Test Recall: {final_recall:.3f}\n")
    f.write(f"Test Precision: {final_precision:.3f}\n")
    f.write(f"Test F1-score: {final_f1_score:.3f}\n")
    f.write(f"Test ROC-AUC: {final_roc_auc:.3f}\n")

print("Model and metadata saved.")

Model and metadata saved.
